# List Fields

Suppose your database stores contractors, and each contractor has a variable-length list of skills. If your database supports array-valued columns, these are very likely stored in such a column rather than in a separate table with a foreign key to contractors.

Now suppose you want to filter for contractors who have two specific skills — will your semantic layer allow that? How about counting the contractors that have each particular skill? Or showing the complete list of skills available from contractors in each city?

SLayer types the field over such a column as `ARRAY` with an `element_type` (recognizing these automatically when ingesting schemas), and gives it semantics of its own:

- the **list functions** `contains`, `contains_any`, `contains_all` and `length` test a whole list;
- the list's **elements** are a child dataset `<field>.value` you can filter, group and aggregate, without fanning out the parent;
- `collect(x)` goes the other way and aggregates values *into* a list.

This notebook uses the two list fields of the Jaffle Shop demo: `products.ingredients` (the supplies a product is made from) and `orders.item_skus` (the SKU of each unit on an order, repeats kept).

**Prerequisites:** `pip install motley-slayer`

In [ ]:
import os
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 120
pd.options.display.width = 250

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.errors import SlayerError
from slayer.core.models import field_type_text

engine, storage, datasets = ensure_jaffle_shop()


def run(query, **kwargs):
    return pd.DataFrame(engine.execute_sync(query, **kwargs).data)


def refused(query):
    try:
        engine.execute_sync(query)
    except SlayerError as error:
        print(error)

## The list type

Ingestion reads each database's native list type (`VARCHAR[]` here; BigQuery `REPEATED`, ClickHouse `Array(T)`, Snowflake `ARRAY`, …) as `datatype: ARRAY` with the element type it holds. The field's type text tells an agent what a list takes:

In [ ]:
by_name = {d.name: d for d in datasets}
for dataset_name, field_name in [("products", "ingredients"), ("orders", "item_skus")]:
    field = by_name[dataset_name].get_field(field_name)
    print(f"{dataset_name}.{field_name}: {field_type_text(field)}")

## Testing a whole list

`contains(list, x)` asks whether `x` is one of the list's elements; `contains_all` / `contains_any` compare it with another list, here an array literal; `length` counts the distinct elements. They work wherever an expression does — filters, computed dimensions, metrics.

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.name", {"field": "length(products.ingredients)", "alias": "ingredient_count"}],
    "filter": ["contains(products.ingredients, 'bread')"],
})

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.name"],
    "filter": ["contains_all(products.ingredients, ['bread', 'cheese'])"],
})

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.name"],
    "filter": ["contains_any(products.ingredients, ['banana', 'chai mix'])"],
})

Absence is a `NOT contains(...)`: the products made without bread.

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.type"],
    "metrics": ["count(products.*)"],
    "filter": ["NOT contains(products.ingredients, 'bread')"],
})

A list-valued `{variable}` in a list argument binds as a list, so a "basket" can come from the caller:

In [ ]:
run(
    {"population": "orders", "metrics": ["count(orders.*)"], "filter": ["contains_any(orders.item_skus, {wanted})"]},
    variables={"wanted": ["JAF-004", "JAF-005"]},
)

## A list as a set

A whole list groups and compares as the set of its distinct elements, shown sorted: products whose ingredients hold the same supplies, repeats and order aside, form one group. A list still has no single value to compare with a scalar or to sum, so those are refused with a typed error that names what to do instead. On a text field, `contains` is the substring test instead.

In [ ]:
run({"population": "products", "dimensions": ["products.ingredients"], "metrics": ["count(products.*)"]})

In [ ]:
refused({"population": "products", "dimensions": ["products.name"], "filter": ["products.ingredients = 'bread'"]})

In [ ]:
refused({"population": "orders", "metrics": ["sum(orders.item_skus)"]})

In [ ]:
run({"population": "products", "dimensions": ["products.name"], "filter": ["contains(products.name, 'stew')"]})

## Elements as rows

Every list field is also a one-to-many hop to its elements: `products.ingredients.value` is one row per non-NULL element, and `count(products.ingredients.*)` counts those rows. Group by the element, and `count(products.*)` is the number of **products** using each ingredient:

The warning is association's usual reminder: a product counts once under each of its ingredients, so the cells overlap and do not add up to the product total.

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.ingredients.value"],
    "metrics": ["count(products.*)"],
})

On orders, `count(orders.item_skus.*)` counts units while the associated `count(orders.*)` counts orders, so the two differ by the orders holding a SKU more than once:

In [ ]:
run({
    "population": "orders",
    "dimensions": ["orders.item_skus.value"],
    "metrics": [
        {"metric": "count(orders.*)", "alias": "orders_with_sku"},
        {"metric": "count(orders.item_skus.*)", "alias": "units"},
    ],
})

## No fan-out

Reading the elements never multiplies the parent rows: each aggregation runs over its own rows, so an order metric next to an element metric keeps its value. The order counts and revenue below are the same with or without the `units` metric beside them.

In [ ]:
with_units = run({
    "population": "orders",
    "dimensions": ["stores.name"],
    "metrics": ["count(orders.*)", "sum(orders.order_total)", {"metric": "count(orders.item_skus.*)", "alias": "units"}],
})
without_units = run({
    "population": "orders",
    "dimensions": ["stores.name"],
    "metrics": ["count(orders.*)", "sum(orders.order_total)"],
})
with_units

In [ ]:
with_units.drop(columns="units").equals(without_units)

## Every element condition reads the same element

All `orders.item_skus.value` conditions in one query constrain **one** element, so asking for a jaffle and a drink with two element filters matches nothing — no single SKU is both. To require several elements, use `contains_all`:

In [ ]:
run({
    "population": "orders",
    "metrics": ["count(orders.*)"],
    "filter": ["orders.item_skus.value = 'JAF-001'", "orders.item_skus.value = 'BEV-001'"],
})

In [ ]:
run({
    "population": "orders",
    "metrics": ["count(orders.*)"],
    "filter": ["contains_all(orders.item_skus, ['JAF-001', 'BEV-001'])"],
})

## Collecting values into a list

`collect(x)` is the aggregation that builds a list: the distinct non-NULL values of `x` in each cell, sorted. It takes fields, elements and expressions alike:

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.type"],
    "metrics": [{"metric": "collect(products.ingredients.value)", "alias": "all_ingredients"}],
})

`collect` works on regular fields too — here, the products of each type:

In [ ]:
run({
    "population": "products",
    "dimensions": ["products.type"],
    "metrics": [{"metric": "collect(products.name)", "alias": "product_names"}],
})

It also collects a field of a joined dataset, one list per parent row. Here are each customer's order statuses, fewest orders first:

In [ ]:
run({
    "population": "customers",
    "dimensions": ["customers.name"],
    "metrics": [
        {"metric": "count(orders.*)", "alias": "order_count"},
        {"metric": "collect(orders.status)", "alias": "statuses"},
    ],
    "order": [{"field": "order_count", "direction": "asc"}, {"field": "customers.name"}],
    "limit": 5,
})

A cell with no values gets the empty list, never `NULL` — here the orders that have no items:

In [ ]:
run({
    "population": "orders",
    "dimensions": [{"field": "length(orders.item_skus)", "alias": "item_count"}],
    "metrics": ["count(orders.*)", {"metric": "collect(orders.item_skus.value)", "alias": "skus"}],
    "filter": ["length(orders.item_skus) < 2"],
})

## Lists across stages

A collected list stays a list in a later stage, with the same functions and elements. The first stage builds each customer's basket of SKUs ever ordered; the second counts the customers whose basket holds `JAF-005`, and how many customers ever bought each SKU:

In [ ]:
baskets = {
    "name": "baskets",
    "population": "orders",
    "dimensions": ["orders.customer_id"],
    "metrics": [{"metric": "collect(orders.item_skus.value)", "alias": "skus"}],
}
run([baskets, {"population": "baskets", "metrics": ["count(baskets.*)"], "filter": ["contains(baskets.skus, 'JAF-005')"]}])

In [ ]:
run([
    baskets,
    {"population": "baskets", "dimensions": ["baskets.skus.value"], "metrics": ["count(baskets.*)"]},
])

## Where lists run

[Database support](../../database-support.md) names the databases that run lists, the list functions and `collect`. See [List fields](../../concepts/queries.md#list-fields) and [Lists](../../concepts/references.md#lists) for the full rules.